In [1]:
# [Célula 1: Imports e Leitura]
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pathlib import Path
import seaborn as sns
import os


In [2]:

sns.set_theme(style="whitegrid", palette="dark")

ROOT = Path.cwd().parent
load_dotenv(ROOT / ".env")  # Carrega variáveis de ambiente do .env
DIR_DATAPROCESSED = ROOT / os.getenv("DIR_DATAPROCESSED")

df = pd.read_parquet(os.path.join(DIR_DATAPROCESSED, "preprocessado.parquet"))
df = pd.DataFrame(df)
df_atual = df[df['periodo'] == df['periodo'].max()]
df_ant = df[df['periodo'] == sorted(df['periodo'].unique())[-2]]


In [3]:

# [Célula 2: Regra 1 - Identificando a Ameaça Real (Volume + Crescimento)]
vol_atual = df_atual.groupby('Empresa')['acessos'].sum()
vol_ant = df_ant.groupby('Empresa')['acessos'].sum()
cresc_pct = ((vol_atual - vol_ant) / vol_ant * 100).fillna(0)

# Quem está roubando mercado? (Critério: Mais de 5k acessos E crescimento > 10%)
ameacas = cresc_pct[(vol_atual > 5000) & (cresc_pct > 10.0)].sort_values(ascending=False)

print("--- ALERTA DE MERCADO: Principais Ameaças Competitivas ---")
if not ameacas.empty:
    for empresa, taxa in ameacas.items():
        print(f"ALERTA: {empresa} cresceu {taxa:.1f}% mantendo volume relevante (>5k).")
else:
    print("Nenhuma empresa combinou alto volume com hipercrescimento neste mês.")

# [Célula 3: Regra 2 - Validação da Modalidade de Cobrança]
# Qual modelo de negócio está tracionando mais?
cobranca = df.groupby(['periodo', 'Modalidade de Cobrança'])['acessos'].sum().unstack()
cobranca_mom = cobranca.pct_change() * 100

print("\n--- Tração por Modalidade de Cobrança (MoM %) ---")
print(cobranca_mom.tail(1).round(2))

vencedor_cobranca = cobranca_mom.iloc[-1].idxmax()
print(f"\nInsight Estratégico: O modelo '{vencedor_cobranca}' está liderando a aquisição de clientes. A estratégia de pacotização deve ser espelhada neste modelo.")

--- ALERTA DE MERCADO: Principais Ameaças Competitivas ---
ALERTA: Telecall cresceu 48.1% mantendo volume relevante (>5k).
ALERTA: Connect Iot Solutions Ltda cresceu 12.8% mantendo volume relevante (>5k).

--- Tração por Modalidade de Cobrança (MoM %) ---
Modalidade de Cobrança  Pré-pago  Pós-pago
periodo                                   
2026-03                      0.0      3.28

Insight Estratégico: O modelo 'Pós-pago' está liderando a aquisição de clientes. A estratégia de pacotização deve ser espelhada neste modelo.
